# 00 — CryptoLens Databricks validation and presentation checks
This notebook runs **inside Databricks**. It checks row uniqueness, timestamp correctness, Gold referential integrity, audit logs and indicator behavior on missing dates. The optional proof cell can actually rerun Silver and compare stable business data instead of comparing two counts without intervening work.

In [ ]:
from pyspark.sql import functions as F, Window

dbutils.widgets.dropdown("run_silver_rerun_proof", "false", ["true", "false"])
dbutils.widgets.text("silver_notebook_path", "")
dbutils.widgets.text("verify_symbol", "BTCUSDT")
dbutils.widgets.text("verify_date", "2026-09-25")
dbutils.widgets.dropdown("verify_load_type", "incremental", ["full", "incremental"])

BRONZE = "workspace.cryptolens.bronze_klines"
SILVER = "workspace.cryptolens.silver_klines"
SILVER_DIM = "workspace.cryptolens.dim_symbol"
GOLD_DIM = "workspace.gold.dim_symbol"
GOLD_FACT = "workspace.gold.fact_daily_ohlcv"
GOLD_VOL = "workspace.gold.volatility_daily"
GOLD_MA = "workspace.gold.moving_averages"
GOLD_CORR = "workspace.gold.symbol_correlation"
AUDIT = "workspace.cryptolens.pipeline_execution_logs"

## 1 — Tables and row counts

In [ ]:
for table in [BRONZE, SILVER, SILVER_DIM, GOLD_DIM, GOLD_FACT, GOLD_VOL, GOLD_MA, GOLD_CORR, AUDIT]:
    if not spark.catalog.tableExists(table):
        raise AssertionError(f"Missing expected table: {table}")
    print(f"{table:52s} {spark.table(table).count():>12,}")

## 2 — Duplicate-key check (actual data, not a claimed count)

In [ ]:
unique_keys = {
    BRONZE: ["symbol", "interval", "load_type", "open_time"],
    SILVER: ["symbol", "interval", "open_time"],
    SILVER_DIM: ["symbol"],
    GOLD_DIM: ["symbol_key"],
    GOLD_FACT: ["symbol_key", "date_key"],
    GOLD_VOL: ["symbol_key", "date_key"],
    GOLD_MA: ["symbol_key", "date_key"],
    GOLD_CORR: ["symbol_a_key", "symbol_b_key", "date_key"],
}
for table, keys in unique_keys.items():
    duplicates = (spark.table(table).groupBy(*keys).count()
                  .filter(F.col("count") > 1).count())
    print(f"duplicate-key groups — {table}: {duplicates}")
    if duplicates:
        raise AssertionError(f"{duplicates} duplicate keys found in {table}")

## 3 — Correct timestamp range and candle quality

In [ ]:
silver_df = spark.table(SILVER)
bad_dates = silver_df.filter(
    F.col("open_time").isNull() |
    (F.col("open_time") < F.lit("2010-01-01").cast("timestamp")) |
    (F.col("open_time") >= F.lit("2100-01-01").cast("timestamp"))).count()
bad_ohlc = silver_df.filter(
    (F.col("high") < F.col("low")) |
    (~F.col("open").between(F.col("low"), F.col("high"))) |
    (~F.col("close").between(F.col("low"), F.col("high")))).count()
print(f"Invalid Silver timestamps: {bad_dates:,}; invalid OHLC: {bad_ohlc:,}")
if bad_dates or bad_ohlc:
    raise AssertionError("Silver validation failed")
display(silver_df.filter(F.col("symbol") == dbutils.widgets.get("verify_symbol"))
        .orderBy("open_time").limit(5))

## 4 — Gold relationships and missing trading dates

In [ ]:
missing_symbol = spark.table(GOLD_FACT).join(spark.table(GOLD_DIM), "symbol_key", "left_anti").count()
missing_date = spark.table(GOLD_FACT).join(spark.table("workspace.gold.dim_date"), "date_key", "left_anti").count()
print("Gold fact rows without matching dim_symbol:", missing_symbol)
print("Gold fact rows without matching dim_date:", missing_date)
if missing_symbol or missing_date:
    raise AssertionError("Gold foreign-key reference failure")

sample = (spark.table(GOLD_FACT)
    .filter(F.col("symbol") == dbutils.widgets.get("verify_symbol"))
    .select("trade_date").orderBy("trade_date")
    .withColumn("previous_date", F.lag("trade_date").over(
        Window.orderBy("trade_date")))
    .withColumn("calendar_gap_days", F.datediff("trade_date", "previous_date"))
    .filter(F.col("calendar_gap_days") > 1))
print("Trading-date gaps for the selected symbol:")
display(sample)

## 5 — No rolling indicator should bridge a calendar gap

In [ ]:
if sample.limit(1).count():
    gap_dates = sample.select("trade_date")
    crossings = (spark.table(GOLD_VOL).filter(F.col("symbol") == dbutils.widgets.get("verify_symbol"))
                 .join(gap_dates, "trade_date")
                 .filter(F.col("daily_return").isNotNull() |
                         F.col("rolling_vol").isNotNull()).count())
    bad_ma = (spark.table(GOLD_MA).filter(F.col("symbol") == dbutils.widgets.get("verify_symbol"))
              .join(gap_dates, "trade_date")
              .filter(F.col("ma_short").isNotNull() | F.col("ma_long").isNotNull()).count())
    print("Gap-day non-null returns/rolling vol:", crossings)
    print("Gap-day non-null moving averages:", bad_ma)
    if crossings or bad_ma:
        raise AssertionError("Rolling metrics incorrectly cross a missing-data gap")

## 6 — Audit trail and quarantine evidence

In [ ]:
display(spark.table(AUDIT).groupBy("layer", "status").count().orderBy("layer", "status"))
display(spark.table(AUDIT).orderBy(F.desc("start_time")).limit(20))
for table in ["workspace.cryptolens.bronze_quarantine", "workspace.cryptolens.silver_quarantine"]:
    if spark.catalog.tableExists(table):
        print(f"{table}: {spark.table(table).count():,} quarantined records")
        display(spark.table(table).limit(5))
    else:
        print(f"{table}: not created — no rejected records have been encountered")

## 7 — Optional REAL idempotency proof
By default this cell does not rerun the pipeline. To demonstrate a real repeat run during assessment, provide the Databricks workspace path to `02_silver_transform` and set `run_silver_rerun_proof=true`. It snapshots a small symbol/date batch, actually reruns the Silver notebook, and verifies identical business values **including** unchanged `load_timestamp`.

In [ ]:
if dbutils.widgets.get("run_silver_rerun_proof") == "true":
    path = dbutils.widgets.get("silver_notebook_path").strip()
    if not path:
        raise ValueError("Provide silver_notebook_path (Databricks workspace notebook path)")
    symbol = dbutils.widgets.get("verify_symbol")
    day = dbutils.widgets.get("verify_date")
    from datetime import date
    d = date.fromisoformat(day)
    def snapshot():
        names = ["symbol", "interval", "open_time", "close_time", "open", "high", "low", "close",
                 "volume", "quote_asset_volume", "number_of_trades", "taker_buy_base_vol",
                 "taker_buy_quote_vol", "load_timestamp"]
        df = (spark.table(SILVER)
              .filter((F.col("symbol") == symbol) &
                      (F.to_date("open_time") == F.lit(d)))
              .select(*names).orderBy("open_time"))
        return [tuple(row) for row in df.collect()]

    before = snapshot()
    if not before:
        raise ValueError("No rows for selected verify_symbol / verify_date")
    print(f"Captured {len(before)} Silver rows; running {path} again")
    args = {"interval": "1m", "load_type": dbutils.widgets.get("verify_load_type"),
            "year": str(d.year), "month": f"{d.month:02d}",
            "day": f"{d.day:02d}" if dbutils.widgets.get("verify_load_type") == "incremental" else ""}
    dbutils.notebook.run(path, 0, args)
    after = snapshot()
    print("Rows before:", len(before), "after:", len(after),
          "identical business values and load_timestamp:", before == after)
    if before != after:
        raise AssertionError("Silver idempotency rerun did not preserve the batch exactly")
    print("[PASS] Real Silver rerun is idempotent for the chosen symbol/date")
else:
    print("Idempotency rerun is OFF (read-only verification mode)")